# DSPy Prompt Optimizer — Template

This notebook automatically finds the best version of your prompt using DSPy.

**What you do:** Fill in 5 config sections (marked ✏️). Run everything else as-is.

**What you get:** A comparison of baseline vs optimized accuracy, plus the exact optimized prompt text you can copy into your production system.

---

**Workflow**
```
✏️  Configure  →  ▶ Run Setup  →  ▶ Optimize  →  📋 Copy prompt
```

> **Prerequisites:** `pip install dspy python-dotenv`  
> An OpenAI API key in a `.env` file or set below.

In [ ]:
# Uncomment and run this cell once if you haven't installed the packages
# %pip install dspy python-dotenv

In [ ]:
import os, random, dspy
from dotenv import load_dotenv
load_dotenv()

# If you're not using a .env file, paste your key here:
# os.environ["OPENAI_API_KEY"] = "sk-..."

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY not found. Set it in .env or uncomment the line above.")

print("Ready.")

---
## ✏️ Section 1 — Configure

**Edit only the cells marked ✏️ below. Everything else can be run as-is.**

### ✏️ Step 1 — Choose your models

| Model | Role | Tip |
|-------|------|-----|
| `INFERENCE_MODEL` | Runs every query at evaluation time | Use a cheap/fast model (e.g. `gpt-4.1-nano`) |
| `OPTIMIZER_MODEL` | Proposes better instruction text | Use a smarter model (e.g. `gpt-4o-mini`) — only used during optimization, not in prod |

Any model supported by LiteLLM works: `openai/gpt-4o-mini`, `anthropic/claude-haiku-4-5`, `ollama/llama3`, etc.

In [ ]:
# ✏️ Edit these two lines
INFERENCE_MODEL = "openai/gpt-4.1-nano"   # runs every query
OPTIMIZER_MODEL = "openai/gpt-4o-mini"    # proposes better instructions

# ── Don't edit below this line ──────────────────────────────────────────────
inference_lm = dspy.LM(model=INFERENCE_MODEL, max_tokens=256, cache=False)
optimizer_lm = dspy.LM(model=OPTIMIZER_MODEL, max_tokens=1024)
dspy.configure(lm=inference_lm)
print(f"Inference : {INFERENCE_MODEL}")
print(f"Optimizer : {OPTIMIZER_MODEL}")

### ✏️ Step 2 — Define your task

A **Signature** tells DSPy:
- What the task is (the docstring — becomes your system prompt)
- What fields go **in** (inputs from your data)
- What fields come **out** (what the model should produce)

**How to edit:**
1. Replace the docstring with your task instruction
2. Add/remove `InputField` lines to match your data columns
3. Add/remove `OutputField` lines to match what you want the model to return

```python
# Example for a sentiment classifier
class MySignature(dspy.Signature):
    """Classify the sentiment of the review."""
    review:    str = dspy.InputField(desc="the customer review text")
    sentiment: str = dspy.OutputField(desc="one of: positive, negative, neutral")
```

In [ ]:
class MySignature(dspy.Signature):
    """Classify the sentiment of the financial news headline.
    Reply with one of: bearish, bullish, neutral."""

    sentence:  str = dspy.InputField(desc="a financial news headline or tweet")
    sentiment: str = dspy.OutputField(desc="one of: bearish, bullish, neutral")


print("Inputs  :", list(MySignature.input_fields.keys()))
print("Outputs :", list(MySignature.output_fields.keys()))

### ✏️ Step 3 — Load your dataset

Each row must include **all input fields and all output fields** you defined above.  
The keys must match the field names in `MySignature` exactly.

**Minimum recommended size:** 30 examples (the more the better).  
The notebook auto-splits into 60% train / 20% val / 20% test.

In [ ]:
from datasets import load_dataset

ds = load_dataset("zeroshot/twitter-financial-news-sentiment", split="train")

LABELS = {0: "bearish", 1: "bullish", 2: "neutral"}

import random
random.seed(42)
sample = random.sample(list(ds), 500)

raw_data = [
    {"sentence": row["text"], "sentiment": LABELS[row["label"]]}
    for row in sample
]

print(f"Loaded {len(raw_data)} examples.")
print(f"Sample: {raw_data[0]['sentence']}")
print(f"Label : {raw_data[0]['sentiment']}")

### ✏️ Step 4 — Choose your metric

The metric scores each prediction so the optimizer knows what "better" means.

| Option | When to use |
|--------|-------------|
| `"exact_match"` | The model must produce the exact expected string (case-insensitive) |
| `"contains"` | The expected answer just needs to appear somewhere in the output |
| `"custom"` | You write your own scoring function below |

In [ ]:
# ✏️ Choose one: "exact_match" | "contains" | "custom"
METRIC = "exact_match"

# ✏️ Only used if METRIC = "custom" — edit the function body
def custom_metric(example, prediction, trace=None):
    # Return 1.0 for correct, 0.0 for wrong (or a float between 0–1)
    expected  = example.category.lower().strip()
    predicted = prediction.category.lower().strip()
    return float(expected == predicted)

print(f"Metric: {METRIC}")

### ✏️ Step 5 — Choose your optimizer

| Optimizer | What it does | Best for |
|-----------|-------------|----------|
| `"bootstrap"` | Finds the best few-shot examples to include in the prompt | Most tasks — start here |
| `"mipro"` | Rewrites the instruction text AND picks few-shot examples | Tasks where your instruction is vague or underspecified |

> **Tip:** If your instruction text is already carefully written, prefer `bootstrap`. `mipro` may rewrite it in ways that hurt performance.

In [ ]:
# ✏️ Choose optimizer: "bootstrap" | "mipro"
OPTIMIZER = "bootstrap"

# ── Bootstrap settings ────────────────────────────────────────────────────────
BOOTSTRAP_CANDIDATES = 10   # ✏️ how many prompt variants to try (more = slower but better)
MAX_DEMOS            = 4    # ✏️ max few-shot examples to include in the prompt

# ── MIPROv2 settings (only used if OPTIMIZER = "mipro") ──────────────────────
MIPRO_AUTO = "medium"       # ✏️ search budget: "light" | "medium" | "heavy"

print(f"Optimizer : {OPTIMIZER}")
if OPTIMIZER == "bootstrap":
    print(f"Candidates: {BOOTSTRAP_CANDIDATES}  |  Max demos: {MAX_DEMOS}")
elif OPTIMIZER == "mipro":
    print(f"Auto budget: {MIPRO_AUTO}")

---
## ▶ Section 2 — Run

**Don't edit these cells — just run them top to bottom.**

In [ ]:
# ── Auto-setup from your configuration ──────────────────────────────────────

input_field_names  = list(MySignature.input_fields.keys())
output_field_names = list(MySignature.output_fields.keys())

# Validate that raw_data has the expected keys
required_keys = set(input_field_names + output_field_names)
missing = [k for row in raw_data for k in required_keys if k not in row]
if missing:
    raise KeyError(f"Dataset rows are missing these fields: {set(missing)}. "
                   f"Check your raw_data keys match your Signature field names.")

# Build DSPy examples
examples = [
    dspy.Example(**row).with_inputs(*input_field_names)
    for row in raw_data
]

random.seed(42)
random.shuffle(examples)

n         = len(examples)
train_end = max(1, int(n * 0.6))
val_end   = max(train_end + 1, int(n * 0.8))

train = examples[:train_end]
val   = examples[train_end:val_end]
test  = examples[val_end:]

if not test:
    raise ValueError("Dataset too small — add more examples so the test split has at least 1 row.")

print(f"Dataset split — Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")

# Build metric function
if METRIC == "exact_match":
    def metric(example, prediction, trace=None):
        return float(all(
            getattr(example, f, "").lower().strip() == getattr(prediction, f, "").lower().strip()
            for f in output_field_names
        ))
elif METRIC == "contains":
    def metric(example, prediction, trace=None):
        return float(all(
            getattr(example, f, "").lower() in getattr(prediction, f, "").lower()
            or getattr(prediction, f, "").lower() in getattr(example, f, "").lower()
            for f in output_field_names
        ))
elif METRIC == "custom":
    metric = custom_metric
else:
    raise ValueError(f"Unknown METRIC: {METRIC!r}. Choose 'exact_match', 'contains', or 'custom'.")

# Build module — Predict for direct output, no reasoning trace
class OptimizationModule(dspy.Module):
    def __init__(self):
        self.predict = dspy.Predict(MySignature)
    def forward(self, **inputs):
        return self.predict(**inputs)

def run_program(program, example):
    return program(**{f: getattr(example, f) for f in input_field_names})

def evaluate(program, dataset, label=""):
    scores = []
    for ex in dataset:
        try:
            pred = run_program(program, ex)
            scores.append(metric(ex, pred))
        except Exception:
            scores.append(0.0)
    n      = len(scores)
    correct = sum(s == 1.0 for s in scores)
    tag    = f"[{label}] " if label else ""
    print(f"{tag}Accuracy: {correct}/{n} ({correct/n:.1%})")
    return correct / n

print("Setup complete.")

In [ ]:
print("── Baseline (no optimization) ──")
baseline_program = OptimizationModule()
baseline_score   = evaluate(baseline_program, test, label="baseline")

In [ ]:
print(f"── Optimizing with: {OPTIMIZER} ──")

if OPTIMIZER == "bootstrap":
    optimizer = dspy.BootstrapFewShotWithRandomSearch(
        metric=metric,
        max_bootstrapped_demos=MAX_DEMOS,
        max_labeled_demos=MAX_DEMOS // 2,
        num_candidate_programs=BOOTSTRAP_CANDIDATES,
        num_threads=4,
    )
    optimized = optimizer.compile(
        OptimizationModule(),
        trainset=train,
        valset=val,
    )

elif OPTIMIZER == "mipro":
    optimizer = dspy.MIPROv2(
        metric=metric,
        auto=MIPRO_AUTO,
        num_threads=4,
        prompt_model=optimizer_lm,
    )
    optimized = optimizer.compile(
        OptimizationModule(),
        trainset=train,
        valset=val,
        requires_permission_to_run=False,
    )

else:
    raise ValueError(f"Unknown OPTIMIZER: {OPTIMIZER!r}. Choose 'bootstrap' or 'mipro'.")

print("Optimization complete.")

In [ ]:
print("── Optimized program ──")
optimized_score = evaluate(optimized, test, label="optimized")

delta = (optimized_score - baseline_score) * 100
print(f"\nResult: {baseline_score:.1%} → {optimized_score:.1%}  ({delta:+.1f}pp)")

---
## 📋 Section 3 — Export your optimized prompt

The cell below runs one example through the optimized program and prints the **exact prompt that was sent to the model** — including the instruction text and any few-shot examples the optimizer selected.

Copy everything from `System:` down to (but not including) the last user input block. That is your production prompt template.

In [ ]:
print("Running one example to capture the prompt...\n")
_ = run_program(optimized, test[0])

print("=" * 70)
print("OPTIMIZED PROMPT — copy this into your production system")
print("=" * 70)
print()
dspy.inspect_history(n=1)
print()
print("=" * 70)
print(f"TIP: The last message block is the live test input.")
print(f"     In production, replace it with your variable, e.g. {{{input_field_names[0]}}}")

### Inspect individual predictions

Run the cell below to see how the optimized program handles specific examples.

In [ ]:
print("── Sample predictions on test set ──\n")
for i, ex in enumerate(test[:5]):
    pred = run_program(optimized, ex)
    score = metric(ex, pred)
    status = "✓" if score == 1.0 else "✗"
    inputs_str = "  ".join(f"{f}={getattr(ex, f)!r}" for f in input_field_names)
    outputs_str = "  ".join(f"{f}={getattr(pred, f, '?')!r}" for f in output_field_names)
    expected_str = "  ".join(f"{f}={getattr(ex, f)!r}" for f in output_field_names)
    print(f"[{i+1}] {status}  Input   : {inputs_str}")
    print(f"       Expected: {expected_str}")
    print(f"       Got     : {outputs_str}")
    print()